# 03 — Lenox Square versus Greenbriar Mall

**What we are doing:** Restrict the four-destination panel to Lenox and Greenbriar. Origin fixed effects control for stable differences across ZCTAs. Interactions ask whether the **relative** Lenox-versus-Greenbriar relationship varies with income, inequality, or neighborhood composition.

$$E[Y_{ijt}|X]=\exp(\alpha_i+\tau_t+\lambda Lenox_j+\beta_d\ln(1+d_{ij})+\delta_x(x_i\times Lenox_j))$$

Do not interpret area demographic shares as the identities or motives of individual shoppers.

In [ ]:
from pathlib import Path
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
PROCESSED=ROOT/'data'/'processed'
scenarios=['2020_geometry_provisional','2010_equal_block_diagnostic']
formulas={
'P1_distance':'flow ~ log_distance + Lenox + C(ZCTA) + C(Month)',
'P2_income_interaction':'flow ~ log_distance + Lenox + log_income:Lenox + C(ZCTA) + C(Month)',
'P3_gini_interaction':'flow ~ log_distance + Lenox + gini:Lenox + C(ZCTA) + C(Month)',
'P4_black_share_interaction':'flow ~ log_distance + Lenox + black_share:Lenox + C(ZCTA) + C(Month)',
'P5_income_and_black_share':'flow ~ log_distance + Lenox + log_income:Lenox + black_share:Lenox + C(ZCTA) + C(Month)'}
rows=[]
for scenario in scenarios:
    panel=pd.read_csv(PROCESSED/f'panel_{scenario}.csv',dtype={'ZCTA':str})
    pair=panel[panel.Mall.isin(['Lenox Square','Greenbriar Mall'])].copy()
    pair['Lenox']=(pair.Mall=='Lenox Square').astype(int)
    for name,formula in formulas.items():
        m=smf.glm(formula,data=pair,family=sm.families.Poisson())
        groups=pair.loc[m.data.row_labels,'ZCTA']
        fit=m.fit(cov_type='cluster',cov_kwds={'groups':groups},maxiter=150)
        for term in ['log_distance','Lenox','log_income:Lenox','gini:Lenox','black_share:Lenox']:
            if term in fit.params:
                rows.append({'scenario':scenario,'model':name,'term':term,'coef':fit.params[term],'se':fit.bse[term],'p':fit.pvalues[term],'n':fit.nobs})
result=pd.DataFrame(rows)
print(result.to_string(index=False))
result.to_csv(PROCESSED/'rerun_lenox_greenbriar.csv',index=False)

## Make a comparison figure
**What we are doing:** Show allocated origin-device counts in each destination by month. These are crosswalk-dependent estimates, not unique mall visitors.

In [ ]:
import matplotlib.pyplot as plt
pair= pd.read_csv(PROCESSED/'panel_2020_geometry_provisional.csv')
monthly=pair[pair.Mall.isin(['Lenox Square','Greenbriar Mall'])].groupby(['Month','Mall']).flow.sum().unstack()
ax=monthly.plot(marker='o',figsize=(10,5))
ax.set(title='Lenox vs Greenbriar: allocated reported origin activity',ylabel='Allocated origin-device count',xlabel='Month')
plt.tight_layout();plt.show()